# DeepAffinity: interpretable compound–protein affinity prediction
## A compact reproduction of Karimi, Wu, Wang & Shen (2019)

This notebook explains and reproduces the central ideas in **“DeepAffinity: interpretable deep learning of compound–protein affinity through unified recurrent and convolutional neural networks”** (*Bioinformatics*, 35(18), 3329–3338).

- [Paper (PubMed Central)](https://pmc.ncbi.nlm.nih.gov/articles/PMC6755266/)
- [Paper DOI](https://doi.org/10.1093/bioinformatics/btz111)
- [Authors' DeepAffinity repository](https://github.com/Shen-Lab/DeepAffinity)

> **Scope.** The original work curates hundreds of thousands of BindingDB measurements, derives protein structural-property sequences (SPS), pretrains sequence autoencoders, compares several architectures, and performs structural interpretation. Here we reproduce the core supervised **joint RNN–CNN + attention** mechanism in modern PyTorch. The notebook is configured for CUDA and automatically uses `data/deepaffinity.csv` when supplied; without that file it runs a clearly labelled synthetic architecture test. This is an architectural reproduction, not a claim to reproduce the paper's reported numbers.

## 1. Problem formulation

Given a compound sequence $c=(c_1,\ldots,c_m)$ and a protein representation $p=(p_1,\ldots,p_n)$, estimate binding affinity $y$:

$$\hat y=f_\theta(c,p).$$

Compounds are represented as **SMILES**. DeepAffinity represents proteins using **SPS strings**, which combine predicted secondary structure, solvent accessibility, and physicochemical class. This reduces raw amino-acid complexity while exposing structural context. Affinities are transformed to a logarithmic scale; for molar measurements a common form is

$$pK=-\log_{10}(K\,[\mathrm M]).$$

Larger $pK$ therefore means stronger binding. The paper emphasizes four challenges: variable-length inputs, scarce labels relative to chemical space, generalization to unseen compounds/proteins, and interpretability.

## 2. What DeepAffinity contributes

The model learns one encoder per modality and joins their representations for regression:

1. **Token embeddings** map SMILES/SPS symbols to dense vectors.
2. **Recurrent layers** capture long-range sequential context. The paper also uses unsupervised seq2seq pretraining; this compact version trains end-to-end.
3. **Convolutions** detect local chemical or structural motifs in contextualized sequences.
4. **Additive attention** computes a normalized importance weight for every position:
   $$e_i=v^\top\tanh(Wh_i+b),\qquad \alpha_i=\frac{\exp e_i}{\sum_j\exp e_j},\qquad z=\sum_i\alpha_i h_i.$$
5. The compound and protein vectors are concatenated and passed through dense layers to predict affinity.

Attention is useful evidence about what the network used, but it is **not automatically a causal explanation** or proof of a physical contact. The paper validates attention against known binding-site information; we inspect whether it recovers planted motifs in our controlled benchmark.

In [ ]:
import copy
import csv
import hashlib
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 17
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
if USE_AMP:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'DeepAffinity' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'PyTorch {torch.__version__} | device: {device} | root: {PROJECT_ROOT}')
if USE_AMP:
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 2**30:.1f} GiB | AMP: enabled')

## 3. Data: real-data hook and reproducible fallback

Place a CSV at `data/deepaffinity.csv` with columns:

```text
smiles,sps,affinity
CC(=O)OC1=CC=CC=C1C(=O)O,<SPS string>,6.23
```

`affinity` should already be a consistent regression target such as $pK_d$, $pK_i$, or $pIC_{50}$; do not silently mix units or assay types. If no file exists, the next cell creates a controlled benchmark in which particular compound and SPS motifs contribute to affinity. The fallback lets us test both prediction and interpretation without pretending synthetic labels are BindingDB measurements.

In [ ]:
def read_real_csv(path):
    with path.open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    required = {'smiles', 'sps', 'affinity'}
    if not rows or not required.issubset(rows[0]):
        raise ValueError(f'{path} must contain columns {sorted(required)}')
    return [(r['smiles'], r['sps'], float(r['affinity'])) for r in rows]

def write_records(path, rows):
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['smiles', 'sps', 'affinity'])
        writer.writerows(rows)

def make_motif_benchmark(n_compounds=80, n_proteins=70, n_pairs=1800, seed=SEED):
    rng = random.Random(seed)
    smiles_alphabet = list('CNOSPFclBr[]=#()123456')
    sps_alphabet = list('HECTSAKLMNQ')  # abstract SPS-like symbols
    compound_motifs = ['C=O', 'NCC', 'c1cc', 'S(=O)', 'C#N']
    protein_motifs = ['HHH', 'EET', 'CCS', 'KLM', 'NQH']

    def sequence(alphabet, length, motif):
        text = ''.join(rng.choice(alphabet) for _ in range(length))
        at = rng.randrange(1, max(2, length - len(motif)))
        return text[:at] + motif + text[at + len(motif):]

    compounds = []
    for i in range(n_compounds):
        motif = compound_motifs[i % len(compound_motifs)]
        compounds.append((sequence(smiles_alphabet, rng.randint(28, 48), motif), i % 5))
    proteins = []
    for i in range(n_proteins):
        motif = protein_motifs[i % len(protein_motifs)]
        proteins.append((sequence(sps_alphabet, rng.randint(55, 90), motif), i % 5))

    interactions = []
    for _ in range(n_pairs):
        ci, pi = rng.randrange(n_compounds), rng.randrange(n_proteins)
        smiles, c_class = compounds[ci]
        sps, p_class = proteins[pi]
        match = 2.2 if c_class == p_class else 0.0
        neighbor = 0.7 if (c_class - p_class) % 5 in (1, 4) else 0.0
        affinity = 5.0 + match + neighbor + rng.gauss(0, 0.18)
        interactions.append((smiles, sps, affinity))
    return interactions, compound_motifs, protein_motifs

DATA_PATH = DATA_DIR / 'deepaffinity.csv'
SYNTHETIC_PATH = DATA_DIR / 'deepaffinity_synthetic.csv'
REAL_DATA = DATA_PATH.exists()
if REAL_DATA:
    records = read_real_csv(DATA_PATH)
    DATA_SOURCE = f'real CSV: {DATA_PATH}'
    COMPOUND_MOTIFS, PROTEIN_MOTIFS = [], []
else:
    if SYNTHETIC_PATH.exists():
        records = read_real_csv(SYNTHETIC_PATH)
    else:
        records, _, _ = make_motif_benchmark()
        write_records(SYNTHETIC_PATH, records)
    COMPOUND_MOTIFS = ['C=O', 'NCC', 'c1cc', 'S(=O)', 'C#N']
    PROTEIN_MOTIFS = ['HHH', 'EET', 'CCS', 'KLM', 'NQH']
    DATA_SOURCE = f'synthetic motif benchmark (architecture test only): {SYNTHETIC_PATH}'

hasher = hashlib.sha256()
for smiles_text, sps_text, affinity in records:
    hasher.update(f'{smiles_text}\t{sps_text}\t{affinity:.12g}\n'.encode())
DATA_FINGERPRINT = hasher.hexdigest()
print(DATA_SOURCE)
print(f'Interactions: {len(records):,}; unique compounds: {len(set(r[0] for r in records))}; unique proteins: {len(set(r[1] for r in records))}')
print('Example:', records[0])

## 4. Split design matters

A random interaction split can place the same compound or protein in both training and test sets, measuring interpolation more than discovery. DeepAffinity therefore examines generalization to unseen entities. Choose:

- `random`: unseen pairs, but usually seen compounds and proteins;
- `compound`: every test compound is absent from training;
- `protein`: every test protein is absent from training.

The default cold-compound split is intentionally harder and guards against one major form of leakage.

In [ ]:
def split_records(rows, mode='compound', train_fraction=0.70, val_fraction=0.15, seed=SEED):
    rng = random.Random(seed)
    if mode == 'random':
        rows = rows.copy(); rng.shuffle(rows)
        n_train, n_val = int(len(rows) * train_fraction), int(len(rows) * val_fraction)
        return rows[:n_train], rows[n_train:n_train+n_val], rows[n_train+n_val:]
    column = 0 if mode == 'compound' else 1
    entities = sorted(set(r[column] for r in rows)); rng.shuffle(entities)
    n_train, n_val = int(len(entities) * train_fraction), int(len(entities) * val_fraction)
    train_ids = set(entities[:n_train])
    val_ids = set(entities[n_train:n_train+n_val])
    test_ids = set(entities[n_train+n_val:])
    return ([r for r in rows if r[column] in train_ids],
            [r for r in rows if r[column] in val_ids],
            [r for r in rows if r[column] in test_ids])

SPLIT_MODE = 'compound'  # 'random', 'compound', or 'protein'
train_rows, val_rows, test_rows = split_records(records, SPLIT_MODE)

class CharacterVocabulary:
    PAD, UNK = '<pad>', '<unk>'
    def __init__(self, texts):
        symbols = sorted(set(''.join(texts)))
        self.itos = [self.PAD, self.UNK] + symbols
        self.stoi = {s: i for i, s in enumerate(self.itos)}
    def encode(self, text, max_length):
        ids = [self.stoi.get(ch, 1) for ch in text[:max_length]]
        return ids + [0] * (max_length - len(ids))
    def __len__(self): return len(self.itos)

# Build vocabularies on training strings only; unknown test symbols map to <unk>.
smiles_vocab = CharacterVocabulary([r[0] for r in train_rows])
sps_vocab = CharacterVocabulary([r[1] for r in train_rows])
MAX_SMILES, MAX_SPS = 128, 256  # inspect truncation before changing these
target_mean = torch.tensor([r[2] for r in train_rows]).mean()
target_std = torch.tensor([r[2] for r in train_rows]).std().clamp_min(1e-6)
print(f'{SPLIT_MODE} split: {len(train_rows)}/{len(val_rows)}/{len(test_rows)}')
print(f'Vocabularies: SMILES={len(smiles_vocab)}, SPS={len(sps_vocab)}')
print(f'Truncated: SMILES={sum(len(r[0]) > MAX_SMILES for r in records)/len(records):.2%}; SPS={sum(len(r[1]) > MAX_SPS for r in records)/len(records):.2%}')

In [ ]:
class AffinityDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        smiles, sps, affinity = self.rows[index]
        return (torch.tensor(smiles_vocab.encode(smiles, MAX_SMILES)),
                torch.tensor(sps_vocab.encode(sps, MAX_SPS)),
                torch.tensor(affinity, dtype=torch.float32))

BATCH_SIZE = 256 if USE_AMP else 64
NUM_WORKERS = min(8, os.cpu_count() or 1) if USE_AMP else 0
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=USE_AMP,
                     persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(AffinityDataset(train_rows), shuffle=True, **loader_kwargs)
val_loader = DataLoader(AffinityDataset(val_rows), **loader_kwargs)
test_loader = DataLoader(AffinityDataset(test_rows), **loader_kwargs)
smiles, sps, affinity = next(iter(train_loader))
print(smiles.shape, sps.shape, affinity.shape)

## 5. Unified recurrent–convolutional model

Each branch uses a bidirectional GRU for global context, a width-3 convolution for local patterns, and masked additive attention. Packed sequences prevent the GRU from processing padding, and padding never receives attention mass. The original implementation and hyperparameters differ; this version uses larger CUDA-oriented encoders while keeping the scientific mechanism visible.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, feature_dim, attention_dim=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(feature_dim, attention_dim), nn.Tanh(),
                                   nn.Linear(attention_dim, 1, bias=False))
    def forward(self, features, mask):
        logits = self.score(features).squeeze(-1).masked_fill(~mask, -1e9)
        weights = torch.softmax(logits, dim=-1)
        summary = torch.sum(features * weights.unsqueeze(-1), dim=1)
        return summary, weights

class RNNCNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.conv = nn.Conv1d(2 * hidden_dim, 2 * hidden_dim, kernel_size=3, padding=1)
        self.attention = AdditiveAttention(2 * hidden_dim)
    def forward(self, token_ids):
        mask = token_ids.ne(0)
        lengths = mask.sum(dim=1).cpu()
        packed = pack_padded_sequence(self.embedding(token_ids), lengths, batch_first=True, enforce_sorted=False)
        contextual, _ = self.rnn(packed)
        contextual, _ = pad_packed_sequence(contextual, batch_first=True, total_length=token_ids.size(1))
        local = torch.relu(self.conv(contextual.transpose(1, 2))).transpose(1, 2)
        local = local * mask.unsqueeze(-1)
        return self.attention(local, mask)

class DeepAffinity(nn.Module):
    def __init__(self, smiles_vocab_size, sps_vocab_size):
        super().__init__()
        self.compound_encoder = RNNCNNEncoder(smiles_vocab_size)
        self.protein_encoder = RNNCNNEncoder(sps_vocab_size)
        self.regressor = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, smiles, sps, return_attention=False):
        compound, compound_attention = self.compound_encoder(smiles)
        protein, protein_attention = self.protein_encoder(sps)
        prediction = self.regressor(torch.cat([compound, protein], dim=-1)).squeeze(-1)
        if return_attention:
            return prediction, compound_attention, protein_attention
        return prediction

model = DeepAffinity(len(smiles_vocab), len(sps_vocab)).to(device)
with torch.no_grad():
    smoke = model(smiles[:4].to(device), sps[:4].to(device))
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}; output: {smoke.shape}')

## 6. Metrics

We report RMSE and Pearson correlation. We also implement the concordance index (CI), used widely for affinity-ranking evaluation:

$$CI=P(\hat y_i>\hat y_j\mid y_i>y_j),$$

with half credit for tied predictions. CI measures ranking, not calibration; RMSE and CI answer different questions.

In [ ]:
mean, std = target_mean.to(device), target_std.to(device)

def concordance_index(target, prediction):
    # Exact pairwise implementation; subsample for very large test sets.
    if len(target) > 5000:
        g = torch.Generator().manual_seed(SEED)
        idx = torch.randperm(len(target), generator=g)[:5000]
        target, prediction = target[idx], prediction[idx]
    dy = target[:, None] - target[None, :]
    dp = prediction[:, None] - prediction[None, :]
    comparable = dy > 0
    if not comparable.any(): return float('nan')
    return ((dp[comparable] > 0).float() + 0.5 * (dp[comparable] == 0).float()).mean().item()

@torch.no_grad()
def evaluate(loader):
    model.eval(); predictions, targets = [], []
    for smiles, sps, target in loader:
        smiles = smiles.to(device, non_blocking=True)
        sps = sps.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred_z = model(smiles, sps)
        predictions.append((pred_z * std + mean).cpu())
        targets.append(target)
    prediction, target = torch.cat(predictions), torch.cat(targets)
    rmse = torch.sqrt(torch.mean((prediction - target) ** 2)).item()
    centered_p, centered_t = prediction - prediction.mean(), target - target.mean()
    pearson = (centered_p * centered_t).sum() / (centered_p.norm() * centered_t.norm()).clamp_min(1e-8)
    return {'rmse': rmse, 'pearson': pearson.item(),
            'ci': concordance_index(target, prediction)}, prediction, target

def train_epoch(optimizer):
    model.train(); total, count = 0.0, 0
    for smiles, sps, target in train_loader:
        smiles = smiles.to(device, non_blocking=True)
        sps = sps.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            loss = nn.functional.mse_loss(model(smiles, sps), (target - mean) / std)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer); scaler.update()
        total += loss.item() * len(target); count += len(target)
    return total / count

In [ ]:
EPOCHS = 100 if REAL_DATA else 25
FORCE_RETRAIN = False
CHECKPOINT_PATH = MODELS_DIR / ('deepaffinity_real.pt' if REAL_DATA else 'deepaffinity_synthetic.pt')
history, best_rmse, start = [], float('inf'), time.time()

if CHECKPOINT_PATH.exists() and not FORCE_RETRAIN:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    compatible = (checkpoint['smiles_vocab'] == smiles_vocab.itos and
                  checkpoint['sps_vocab'] == sps_vocab.itos and
                  checkpoint['max_lengths'] == (MAX_SMILES, MAX_SPS) and
                  checkpoint['data_fingerprint'] == DATA_FINGERPRINT)
    if not compatible:
        raise ValueError('Checkpoint vocabulary/configuration does not match this data; set FORCE_RETRAIN=True.')
    model.load_state_dict(checkpoint['model_state'])
    best_rmse = checkpoint['best_rmse']
    print(f'Loaded reusable checkpoint: {CHECKPOINT_PATH}')
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5, fused=USE_AMP)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    stale_epochs, patience = 0, 15
    for epoch in range(1, EPOCHS + 1):
        train_mse = train_epoch(optimizer)
        val_metrics, _, _ = evaluate(val_loader)
        scheduler.step(val_metrics['rmse'])
        history.append((train_mse, val_metrics['rmse']))
        if val_metrics['rmse'] < best_rmse:
            best_rmse, stale_epochs = val_metrics['rmse'], 0
            torch.save({'model_state': model.state_dict(), 'best_rmse': best_rmse,
                        'smiles_vocab': smiles_vocab.itos, 'sps_vocab': sps_vocab.itos,
                        'max_lengths': (MAX_SMILES, MAX_SPS), 'data_fingerprint': DATA_FINGERPRINT,
                        'target_mean': target_mean.item(), 'target_std': target_std.item()}, CHECKPOINT_PATH)
        else:
            stale_epochs += 1
        print(f"Epoch {epoch:02d} | train MSE(z) {train_mse:.4f} | "
              f"val RMSE {val_metrics['rmse']:.4f} | val CI {val_metrics['ci']:.3f} | lr {optimizer.param_groups[0]['lr']:.2e}")
        if stale_epochs >= patience:
            print(f'Early stopping after {patience} epochs without improvement.')
            break
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
test_metrics, test_prediction, test_target = evaluate(test_loader)
print('Test:', {k: round(v, 4) for k, v in test_metrics.items()})
print(f'Elapsed: {(time.time()-start)/60:.1f} min')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([h[0] for h in history], label='train MSE (z)')
axes[0].plot([h[1] for h in history], label='validation RMSE')
axes[0].set(xlabel='epoch', title='Learning curves'); axes[0].legend()
axes[1].scatter(test_target, test_prediction, s=10, alpha=.45)
lo = min(test_target.min(), test_prediction.min()).item()
hi = max(test_target.max(), test_prediction.max()).item()
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1)
axes[1].set(xlabel='true affinity', ylabel='predicted affinity',
            title=f'{SPLIT_MODE} test split')
plt.tight_layout(); plt.show()

## 7. Interpreting attention

The next cell colors each character by attention weight. On the synthetic benchmark, useful attention should often overlap a planted motif. With real data, high-attention regions are hypotheses for chemical substructures or protein regions worth checking against structural/contact annotations—not conclusions by themselves.

In [ ]:
def attention_for_record(record):
    smiles_text, sps_text, affinity = record
    smiles_ids = torch.tensor([smiles_vocab.encode(smiles_text, MAX_SMILES)]).to(device)
    sps_ids = torch.tensor([sps_vocab.encode(sps_text, MAX_SPS)]).to(device)
    model.eval()
    with torch.no_grad():
        pred_z, a_smiles, a_sps = model(smiles_ids, sps_ids, return_attention=True)
    prediction = (pred_z * std + mean).item()
    return prediction, a_smiles[0, :len(smiles_text)].cpu(), a_sps[0, :len(sps_text)].cpu()

def plot_attention(text, weights, title, width=60):
    text = text[:len(weights)]
    columns = min(width, len(text)); rows = math.ceil(len(text) / columns)
    grid = torch.full((rows, columns), float('nan'))
    grid.view(-1)[:len(weights)] = weights
    fig, ax = plt.subplots(figsize=(min(15, columns * .25 + 2), rows * .55 + 1.2))
    ax.imshow(grid, cmap='YlOrRd', aspect='auto')
    for i, ch in enumerate(text):
        ax.text(i % columns, i // columns, ch, ha='center', va='center', fontsize=9)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); plt.tight_layout(); plt.show()

example = test_rows[0]
prediction, smiles_attention, sps_attention = attention_for_record(example)
print(f'True affinity={example[2]:.3f}; predicted={prediction:.3f}')
plot_attention(example[0], smiles_attention, 'Compound attention (SMILES)')
plot_attention(example[1], sps_attention, 'Protein attention (SPS)')

In [ ]:
def top_positions(text, weights, k=8):
    k = min(k, len(text))
    indices = torch.topk(weights, k).indices.tolist()
    return [(i, text[i], round(weights[i].item(), 4)) for i in indices]

print('Top compound positions:', top_positions(example[0], smiles_attention))
print('Top SPS positions:', top_positions(example[1], sps_attention))
if COMPOUND_MOTIFS:
    print('Planted compound motifs present:', [m for m in COMPOUND_MOTIFS if m in example[0]])
    print('Planted protein motifs present:', [m for m in PROTEIN_MOTIFS if m in example[1]])

## 8. What was—and was not—reproduced

### Reproduced

- paired sequence learning from a compound representation and a protein structural-property representation;
- unified recurrent and convolutional encoders;
- attention-based sequence aggregation and visual interpretation;
- affinity regression with RMSE, Pearson correlation, and concordance index;
- cold-entity splits to test generalization beyond memorized compounds or proteins.

### Deliberately omitted or simplified

- BindingDB curation, censoring rules, measurement harmonization, and the paper's exact train/test sets;
- generation of SPS from predicted protein annotations;
- unsupervised seq2seq pretraining on large unlabeled SMILES and SPS corpora;
- exact legacy TensorFlow architecture, hyperparameter search, ensembles, baselines, and structural case studies.

For a closer replication, obtain the authors' processed splits/SPS data, preserve their compound/protein holdout definitions, add encoder pretraining, run multiple seeds, and compare to the paper's baselines under exactly the same records. Most importantly, report how censored affinities (`>`, `<`) and duplicated measurements are handled—those decisions can move results more than a decorative model tweak.

## References

1. Karimi M, Wu D, Wang Z, Shen Y. *DeepAffinity: interpretable deep learning of compound–protein affinity through unified recurrent and convolutional neural networks.* Bioinformatics. 2019;35(18):3329–3338.
2. Liu T, Lin Y, Wen X, Jorissen RN, Gilson MK. *BindingDB: a web-accessible database of experimentally determined protein–ligand binding affinities.* Nucleic Acids Research. 2007.
3. Bahdanau D, Cho K, Bengio Y. *Neural machine translation by jointly learning to align and translate.* ICLR 2015.